# Pulizia dataset videogiochi vintage

Questo notebook carica il dataset originale dei videogiochi vintage, esegue alcune operazioni di pulizia di base
e salva una versione pulita da usare nel chatbot Rasa.



In [1]:
import pandas as pd
from pathlib import Path


In [ ]:
# Caricamento del file CSV usando il percorso già definito 'csv_path' e una codifica compatibile
csv_path = 'Managerial_and_Decision_Economics_2013_Video_Games_Dataset.csv'
# (evita UnicodeDecodeError cambiando l'encoding a 'latin-1' / 'cp1252' se il file non è UTF-8)
df = pd.read_csv(csv_path, encoding='latin-1', low_memory=False)
print(df.shape)
df.head()

['Nintendo DS' 'Sony PSP' 'X360' 'Nintendo Wii' 'PlayStation 3']


In [10]:
# Colonne one-hot dei publisher
publisher_onehot_cols = [
    "2K", "Acclaim", "Activision", "Atari", "Capcom", "Disney", "Eidos",
    "EA", "Infograme", "Konami", "Microsoft", "Midway", "Namco", "Nintendo",
    "Rockstar", "Sony", "Sega", "THQ", "SquareEnix", "Ubisoft"
]

# Nel caso il CSV avesse spazi strani nei nomi:
df.columns = df.columns.str.strip()

# Filtra solo le colonne che ESISTONO davvero nel dataframe
publisher_cols = [c for c in publisher_onehot_cols if c in df.columns]

# Matrice booleana: True dove c'è un 1
pub_mask = df[publisher_cols].eq(1)

# Nome del publisher derivato dai one-hot:
#   - idxmax() prende la colonna con valore "max" (qui 0 o 1, quindi quella col 1)
#   - se la riga ha tutti 0, `pub_mask.any(axis=1)` è False
publisher_from_onehot = pub_mask.idxmax(axis=1)

# Ora creiamo una colonna "PublisherCompatto"
df["PublisherCompatto"] = publisher_from_onehot.where(
    pub_mask.any(axis=1),  # condizione: almeno un 1 tra le colonne one-hot
    df["Publisher"]        # altrimenti mantieni il Publisher originale
)

# Se vuoi SOVRASCRIVERE direttamente Publisher:
df["Publisher"] = df["PublisherCompatto"]
df = df.drop(columns=["PublisherCompatto"])
df.head()

,Console,Title,US Sales (millions),Block4,Block2,Block1,Block0.5,YearReleased,2004,2005,...,Tricks,Volleyball,Wakeboarding,Wrestling,FirstPerson,Platform,Isometric,SideScrolling,TopDown,ThirdPerson
0,Nintendo DS,Super Mario 64 DS,4.69,1,1,1,1,2004,1,0,...,0,0,0,0,0,1,0,0,0,1
1,Sony PSP,Lumines: Puzzle Fusion,0.56,0,0,0,1,2004,1,0,...,0,0,0,0,0,0,0,0,0,0
2,Nintendo DS,WarioWare Touched!,0.54,0,0,0,1,2004,1,0,...,0,0,0,0,0,1,1,1,1,1
3,Sony PSP,Hot Shots Golf: Open Tee,0.49,0,0,0,0,2004,1,0,...,0,0,0,0,0,0,0,0,0,1
4,Nintendo DS,Spider-Man 2,0.45,0,0,0,0,2004,1,0,...,0,0,0,0,0,1,0,1,0,1


In [11]:

# Pulizia delle colonne non necessarie
columns_to_drop = ['Block4', 'Block2', 'Block1', 'Block0.5', '2004', '2005', '2006', '2007', '2008', '2009', '2010', 'YearReleasedSq', 
                   'ReviewSq', 'lnUsedPrice', 'LifecycleSq', 'MaxPlayersSq', 'GBA','GCN','NDS','Wii','PS2','PS3','PSP','Xbox','X360', 
                   'Lifecycle', 'Licensed', 'Accessory', 'LtdEdition', 'Handheld'
                   ]
df.drop(columns=columns_to_drop, inplace=True)
df.drop(columns=publisher_onehot_cols, inplace=True)
df.head()


,Console,Title,US Sales (millions),YearReleased,Publisher,Genre,Sequel,Re-release,Usedprice,Review Score,...,Tricks,Volleyball,Wakeboarding,Wrestling,FirstPerson,Platform,Isometric,SideScrolling,TopDown,ThirdPerson
0,Nintendo DS,Super Mario 64 DS,4.69,2004,Nintendo,Action,1,1,24.95,85,...,0,0,0,0,0,1,0,0,0,1
1,Sony PSP,Lumines: Puzzle Fusion,0.56,2004,Ubisoft,Strategy,0,0,14.95,89,...,0,0,0,0,0,0,0,0,0,0
2,Nintendo DS,WarioWare Touched!,0.54,2004,Nintendo,"Action, Racing / Driving, Sports",1,0,22.95,81,...,0,0,0,0,0,1,1,1,1,1
3,Sony PSP,Hot Shots Golf: Open Tee,0.49,2004,Sony,Sports,0,0,12.95,81,...,0,0,0,0,0,0,0,0,0,1
4,Nintendo DS,Spider-Man 2,0.45,2004,Activision,Action,1,0,14.95,61,...,0,0,0,0,0,1,0,1,0,1


In [12]:
# 1. Selezioniamo tutte le colonne di "tag di genere"
#    usiamo l'intervallo da "Action" a "ThirdPerson" (inclusa)
start = df.columns.get_loc("Action")
end = df.columns.get_loc("ThirdPerson")
genre_flag_cols = list(df.columns[start:end + 1])

print("Numero colonne tag:", len(genre_flag_cols))
print("Prime colonne tag:", genre_flag_cols[:10])

# 2. Funzione che raccoglie tutti i tag con valore 1 in una stringa
def collect_subgenres(row):
    tags = [col for col in genre_flag_cols if row.get(col) == 1]
    if not tags:
        return None  # oppure "" se preferisci stringa vuota
    # uso il separatore '|' per evitare casini con la virgola nel CSV
    return "|".join(tags)

# 3. Creiamo la nuova colonna
df["SubGenres"] = df.apply(collect_subgenres, axis=1)

df.drop(columns=genre_flag_cols, inplace=True)

# 4. Facciamo un check
df[["Title", "Genre", "SubGenres"]].head(10)

Numero colonne tag: 100
Prime colonne tag: ['Action', 'Adventure', 'Educational', 'Racing', 'RPG', 'Simulation', 'Sports', 'Strategy', 'Adult', 'Anime']


,Title,Genre,SubGenres
0,Super Mario 64 DS,Action,Action|Puzzle|RealTime|Platform|ThirdPerson
1,Lumines: Puzzle Fusion,Strategy,Strategy|Puzzle|RhythmAction
2,WarioWare Touched!,"Action, Racing / Driving, Sports",Action|Racing|Sports|Anime|Paddle|Boxing|Fishi...
3,Hot Shots Golf: Open Tee,Sports,Sports|Golf|ThirdPerson
4,Spider-Man 2,Action,Action|Comics|Fighting|Puzzle|RealTime|Platfor...
5,The Urbz: Sims in the City,Simulation,Simulation|Managerial|Isometric
6,Ridge Racer,Racing / Driving,Racing|FirstPerson|ThirdPerson
7,Metal Gear Ac!d,Strategy,Strategy|SciFi|Spy|TurnBased|ThirdPerson
8,Madden NFL 2005,Sports,Sports|AmericanFootball|ThirdPerson
9,Pokemon Dash,Racing / Driving,Racing|TopDown|ThirdPerson


In [13]:
# pulizia delle colonne di rating
rating_cols = ["RatingE", "RatingT", "RatingM"]

# Per sicurezza, ci assicuriamo che esistano
rating_cols = [c for c in rating_cols if c in df.columns]

def derive_age_rating(row):
    if row.get("RatingM", 0) == 1:
        return "M"  # Mature
    if row.get("RatingT", 0) == 1:
        return "T"  # Teen
    if row.get("RatingE", 0) == 1:
        return "E"  # Everyone
    return "Unknown"

df["AgeRating"] = df.apply(derive_age_rating, axis=1)

df.drop(columns=rating_cols, inplace=True)

df[["Title", "AgeRating"]].head(10)


,Title,AgeRating
0,Super Mario 64 DS,E
1,Lumines: Puzzle Fusion,Unknown
2,WarioWare Touched!,E
3,Hot Shots Golf: Open Tee,Unknown
4,Spider-Man 2,E
5,The Urbz: Sims in the City,E
6,Ridge Racer,E
7,Metal Gear Ac!d,M
8,Madden NFL 2005,E
9,Pokemon Dash,E


In [14]:
# Salvataggio del dataset pulito
output_path = Path('vintage_games_clean.csv')
df.to_csv(output_path, index=False)
print('File salvato in:', output_path.resolve())


File salvato in: C:\Users\matte\Documents\GitHub\DataScience\Chatbot\dataset\vintage_games_clean.csv
